# Champions League Predictor — Part 1: Features

Run this first. Turns the raw CSVs into data the model can use.

Main rule: a match can only use info from **before** it was played. Easy to get wrong.

In [18]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import unicodedata
import joblib

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

## Load the data

In [19]:
games = pd.read_csv('Data_Sources/games.csv', encoding='utf-8')
game_lineups = pd.read_csv('Data_Sources/game_lineups.csv')
players = pd.read_csv('Data_Sources/players.csv')
player_valuations = pd.read_csv('Data_Sources/player_valuations.csv')
competitions = pd.read_csv('Data_Sources/competitions.csv')

games.shape

(88958, 23)

## Check: is league position cheating?

It was my top feature by miles, which made me suspicious. Turns out it's missing for every
Champions League game, so it was useless where I needed it. Dropped it.

In [20]:
# 2025-26 UCL Final is absent from the Kaggle dump — added manually.
# NOTE: 1-1 is the score after extra time; PSG won 4-3 on penalties.
# The schema has no penalty column, so this row's label is 'draw' and CANNOT be
# predicted correctly. Treat the Final as a probability question, not a label match.
final_row = pd.DataFrame([{
    'game_id': 9999999,
    'competition_id': 'CL',
    'season': 2025,
    'round': 'Final',
    'date': '2026-05-30',
    'home_club_id': 583,   # Paris Saint-Germain
    'away_club_id': 11,    # Arsenal FC
    'home_club_goals': 1,
    'away_club_goals': 1,
    'home_club_position': None,
    'away_club_position': None,
    'home_club_manager_name': 'Luis Enrique',
    'away_club_manager_name': 'Mikel Arteta',
    'stadium': 'Puskás Aréna',
    'attendance': 61035,
    'referee': 'Daniel Siebert',
    'url': None,
    'home_club_formation': None,
    'away_club_formation': None,
    'home_club_name': 'Paris Saint-Germain',
    'away_club_name': 'Arsenal FC',
    'aggregate': None,
    'competition_type': 'international_cup'
}])

games = pd.concat([games, final_row], ignore_index=True)

In [21]:
valid_ids = competitions.loc[competitions['confederation'] == 'europa', 'competition_id']
games = games[games['competition_id'].isin(valid_ids)]
games = games.drop(columns=['attendance', 'stadium', 'referee', 'aggregate', 'url'])

games['date'] = pd.to_datetime(games['date'])
games = games.sort_values('date').reset_index(drop=True)
print(games.shape)

(83811, 18)


## Club Elo

Rating that goes up when you win, down when you lose. Stored *before* each match so it doesn't
leak the result.

In [22]:
pos_check = (
    games.dropna(subset=['home_club_position'])
         .groupby(['home_club_id', 'season'])['home_club_position']
         .nunique()
)
print(f"(club, season) groups with a single constant position: {(pos_check == 1).mean():.1%}")
print(f"median distinct positions per club-season:            {pos_check.median():.0f}")

cl = games[games['competition_id'] == 'CL']
print(f"\nposition missing on Champions League matches: {cl['home_club_position'].isna().mean():.1%}")

(club, season) groups with a single constant position: 1.1%
median distinct positions per club-season:            6

position missing on Champions League matches: 100.0%


In [23]:
USE_CLUB_POSITION = False   # flip to True to reproduce v1 behaviour

## Recent form

Points per game over the last 5 and 10 games, goals scored/conceded, days of rest.
Shifted by one game so a match can't see its own result.

In [24]:
K = 20
HOME_ADV = 60
INITIAL_ELO = 1500

elo = {}
home_elo_before, away_elo_before = [], []

for row in games.itertuples(index=False):
    r_home = elo.get(row.home_club_id, INITIAL_ELO)
    r_away = elo.get(row.away_club_id, INITIAL_ELO)

    home_elo_before.append(r_home)
    away_elo_before.append(r_away)

    exp_home = 1 / (1 + 10 ** ((r_away - r_home - HOME_ADV) / 400))

    if row.home_club_goals > row.away_club_goals:
        actual_home = 1.0
    elif row.home_club_goals < row.away_club_goals:
        actual_home = 0.0
    else:
        actual_home = 0.5

    elo[row.home_club_id] = r_home + K * (actual_home - exp_home)
    elo[row.away_club_id] = r_away + K * ((1 - actual_home) - (1 - exp_home))

games['home_elo'] = home_elo_before
games['away_elo'] = away_elo_before

## Manager Elo

Same thing but for managers.

In [25]:
def normalize_name(name):
    if pd.isna(name):
        return name
    return unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('utf-8').strip().lower()

games = games.dropna(subset=['home_club_manager_name', 'away_club_manager_name'])
games = games.sort_values('date').reset_index(drop=True)

games['home_manager_norm'] = games['home_club_manager_name'].apply(normalize_name)
games['away_manager_norm'] = games['away_club_manager_name'].apply(normalize_name)

manager_elo = {}
INITIAL_MANAGER_ELO = 1500
MANAGER_K = 20

hm_before, am_before = [], []

for row in games.itertuples(index=False):
    r_home = manager_elo.get(row.home_manager_norm, INITIAL_MANAGER_ELO)
    r_away = manager_elo.get(row.away_manager_norm, INITIAL_MANAGER_ELO)

    hm_before.append(r_home)
    am_before.append(r_away)

    # no home bonus here, that belongs to the club not the manager
    exp_home = 1 / (1 + 10 ** ((r_away - r_home) / 400))

    if row.home_club_goals > row.away_club_goals:
        actual_home = 1.0
    elif row.home_club_goals < row.away_club_goals:
        actual_home = 0.0
    else:
        actual_home = 0.5

    manager_elo[row.home_manager_norm] = r_home + MANAGER_K * (actual_home - exp_home)
    manager_elo[row.away_manager_norm] = r_away + MANAGER_K * ((1 - actual_home) - (1 - exp_home))

games['home_manager_elo'] = hm_before
games['away_manager_elo'] = am_before

In [ ]:
home_side = games[['game_id', 'date', 'season', 'home_club_id',
                   'home_club_goals', 'away_club_goals']].copy()
home_side.columns = ['game_id', 'date', 'season', 'club_id', 'goals_for', 'goals_against']

away_side = games[['game_id', 'date', 'season', 'away_club_id',
                   'away_club_goals', 'home_club_goals']].copy()
away_side.columns = ['game_id', 'date', 'season', 'club_id', 'goals_for', 'goals_against']

tm = pd.concat([home_side, away_side], ignore_index=True)
tm['points'] = np.select(
    [tm['goals_for'] > tm['goals_against'], tm['goals_for'] == tm['goals_against']],
    [3, 1], default=0
)
tm = tm.sort_values(['club_id', 'date']).reset_index(drop=True)

# shift(1), strictly past matches only
tm['points_prev'] = tm.groupby('club_id')['points'].shift(1)
tm['gf_prev'] = tm.groupby('club_id')['goals_for'].shift(1)
tm['ga_prev'] = tm.groupby('club_id')['goals_against'].shift(1)

for w in (5, 10):
    tm[f'form_ppg_{w}'] = tm.groupby('club_id')['points_prev'].transform(
        lambda s: s.rolling(w, min_periods=1).mean())
    tm[f'form_gf_{w}'] = tm.groupby('club_id')['gf_prev'].transform(
        lambda s: s.rolling(w, min_periods=1).mean())
    tm[f'form_ga_{w}'] = tm.groupby('club_id')['ga_prev'].transform(
        lambda s: s.rolling(w, min_periods=1).mean())

tm['season_ppg'] = tm.groupby(['club_id', 'season'])['points'].transform(
    lambda s: s.shift(1).expanding().mean())
tm['season_played'] = tm.groupby(['club_id', 'season']).cumcount()

# fixture congestion
tm['days_rest'] = tm.groupby('club_id')['date'].diff().dt.days

form_cols = ([f'form_{k}_{w}' for w in (5, 10) for k in ('ppg', 'gf', 'ga')]
             + ['season_ppg', 'season_played', 'days_rest'])

tm[['game_id', 'club_id'] + form_cols].head()

,game_id,club_id,form_ppg_5,form_gf_5,form_ga_5,form_ppg_10,form_gf_10,form_ga_10,season_ppg,season_played,days_rest
0,2221775,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
1,2320460,1,0.0,0.000000,5.000000,0.0,0.000000,5.000000,NaN,0,350.0
2,2361069,1,1.5,1.500000,3.000000,1.5,1.500000,3.000000,3.0,1,52.0
3,2370752,1,2.0,1.666667,2.333333,2.0,1.666667,2.333333,3.0,2,69.0
4,2864716,1,1.5,1.250000,2.250000,1.5,1.250000,2.250000,NaN,0,1349.0


## Lineup stability

How much the starting 11 changed from last game. Rotation = disruption.

In [27]:
form = tm[['game_id', 'club_id'] + form_cols]

games = games.merge(
    form, left_on=['game_id', 'home_club_id'], right_on=['game_id', 'club_id'], how='left'
).rename(columns={c: f'home_{c}' for c in form_cols}).drop(columns='club_id')

games = games.merge(
    form, left_on=['game_id', 'away_club_id'], right_on=['game_id', 'club_id'], how='left'
).rename(columns={c: f'away_{c}' for c in form_cols}).drop(columns='club_id')

print(games.filter(like='form_ppg').isna().mean())

home_form_ppg_5     0.018596
home_form_ppg_10    0.018596
away_form_ppg_5     0.014944
away_form_ppg_10    0.014944
dtype: float64


## Squad value

Market value of the players who actually started. Only uses valuations from before the match.

In [28]:
def parse_leg(r):
    s = str(r).lower()
    if '1st leg' in s:
        return 1
    if '2nd leg' in s:
        return 2
    return 0

games['leg'] = games['round'].apply(parse_leg)

# stage name with the leg suffix stripped, so both legs share a key
games['tie_stage'] = (games['round'].astype(str).str.lower()
                      .str.replace(r'\s*(1st|2nd)\s*leg\s*', '', regex=True)
                      .str.strip())

pair = np.sort(games[['home_club_id', 'away_club_id']].values, axis=1)
games['tie_key'] = (games['competition_id'].astype(str) + '|'
                    + games['season'].astype(str) + '|'
                    + games['tie_stage'] + '|'
                    + pair[:, 0].astype(str) + '|' + pair[:, 1].astype(str))

print(games['leg'].value_counts().to_dict())

{0: 75366, 1: 3807, 2: 3802}


## Missing data flags

Some games have no lineup data. Flag them instead of pretending.

In [29]:
first_legs = (games.loc[games['leg'] == 1,
                        ['tie_key', 'home_club_id', 'home_club_goals', 'away_club_goals', 'date']]
              .rename(columns={'home_club_id': 'fl_home_id',
                               'home_club_goals': 'fl_home_goals',
                               'away_club_goals': 'fl_away_goals',
                               'date': 'fl_date'})
              .drop_duplicates('tie_key'))

games = games.merge(first_legs, on='tie_key', how='left')

is_second = games['leg'] == 2
has_first = is_second & games['fl_home_id'].notna()

# flip to the perspective of the club hosting the SECOND leg
same_side = games['home_club_id'] == games['fl_home_id']
gf = np.where(same_side, games['fl_home_goals'], games['fl_away_goals'])
ga = np.where(same_side, games['fl_away_goals'], games['fl_home_goals'])

games['first_leg_gf'] = np.where(has_first, gf, 0.0)
games['first_leg_ga'] = np.where(has_first, ga, 0.0)
games['agg_lead']     = games['first_leg_gf'] - games['first_leg_ga']

games['is_second_leg'] = is_second.astype(int)
games['is_two_legged'] = (games['leg'] > 0).astype(int)
games['is_knockout'] = games['round'].astype(str).str.lower().str.contains(
    'final|last 16|quarter|semi|round of|leg', regex=True).astype(int)

# sanity: second legs must come after their first leg
bad = games.loc[has_first & (games['date'] <= games['fl_date'])]
print(f"second legs not after their first leg: {len(bad)}")
print(games.loc[is_second, 'agg_lead'].describe())

tie_cols = ['first_leg_gf', 'first_leg_ga', 'agg_lead']
tie_flags = ['is_second_leg', 'is_two_legged', 'is_knockout']

second legs not after their first leg: 1
count    3802.000000
mean       -0.306681
std         1.885321
min        -9.000000
25%        -1.000000
50%         0.000000
75%         1.000000
max         7.000000
Name: agg_lead, dtype: float64


## Difference features

home minus away. Saves the model the work.

In [30]:
game_lineups['date'] = pd.to_datetime(game_lineups['date'])
game_lineups = game_lineups.sort_values('date')
game_lineups = game_lineups.dropna(subset=['player_id', 'club_id', 'game_id', 'position'])

squad_sets = (
    game_lineups
    .groupby(['club_id', 'game_id', 'date'])['player_id']
    .apply(set)
    .reset_index()
    .rename(columns={'player_id': 'squad'})
    .sort_values(['club_id', 'date'])
    .reset_index(drop=True)
)

def jaccard(a, b):
    if not isinstance(a, set) or not isinstance(b, set) or not a or not b:
        return np.nan
    return len(a & b) / len(a | b)

squad_sets['prev_squad'] = squad_sets.groupby('club_id')['squad'].shift(1)
squad_sets['lineup_stability'] = [
    jaccard(a, b) for a, b in zip(squad_sets['squad'], squad_sets['prev_squad'])
]

## Target

home_win / draw / away_win

In [31]:
for side in ('home', 'away'):
    games = games.merge(
        squad_sets[['game_id', 'club_id', 'lineup_stability']],
        left_on=['game_id', f'{side}_club_id'],
        right_on=['game_id', 'club_id'], how='left'
    ).rename(columns={'lineup_stability': f'{side}_lineup_stability'}).drop(columns='club_id')

# game_lineups has nothing before 2013-07-01
games = games[games['date'] >= '2013-07-01'].reset_index(drop=True)

## Build the feature matrix

Dropped `season` — training is old seasons, testing is new ones, so it just confuses the model.

In [32]:
player_valuations['date'] = pd.to_datetime(player_valuations['date'])

lineup_with_value = pd.merge_asof(
    game_lineups.sort_values('date'),
    player_valuations.sort_values('date')[['player_id', 'date', 'market_value_in_eur']],
    on='date', by='player_id', direction='backward'
)

squad_value = (
    lineup_with_value
    .groupby(['game_id', 'club_id'])['market_value_in_eur']
    .agg(total_squad_value='sum', avg_squad_value='mean')
    .reset_index()
)

for side in ('home', 'away'):
    games = games.merge(
        squad_value, left_on=['game_id', f'{side}_club_id'],
        right_on=['game_id', 'club_id'], how='left'
    ).rename(columns={'total_squad_value': f'{side}_squad_value',
                      'avg_squad_value': f'{side}_avg_value'}).drop(columns='club_id')

## Split by date

No shuffling. Train on the past, test on the future.

In [33]:
long = pd.concat([
    games[['date','home_club_id','home_squad_value','home_avg_value']].rename(
        columns={'home_club_id':'club_id','home_squad_value':'sv','home_avg_value':'av'}),
    games[['date','away_club_id','away_squad_value','away_avg_value']].rename(
        columns={'away_club_id':'club_id','away_squad_value':'sv','away_avg_value':'av'}),
]).dropna(subset=['sv']).sort_values('date')

games = games.sort_values('date').reset_index(drop=True)

for side in ('home','away'):
    games = pd.merge_asof(
        games, long.rename(columns={'sv':f'{side}_sv_last','av':f'{side}_av_last'}),
        on='date', left_by=f'{side}_club_id', right_by='club_id',
        direction='backward', allow_exact_matches=False
    ).drop(columns='club_id')
    games[f'{side}_squad_value'] = games[f'{side}_squad_value'].fillna(games[f'{side}_sv_last'])
    games[f'{side}_avg_value']   = games[f'{side}_avg_value'].fillna(games[f'{side}_av_last'])
    games = games.drop(columns=[f'{side}_sv_last', f'{side}_av_last'])

## Fill missing values

Using training data only.

In [34]:
flag_sources = ['home_squad_value', 'home_avg_value', 'away_squad_value', 'away_avg_value',
                'home_lineup_stability', 'away_lineup_stability',
                'home_season_ppg', 'away_season_ppg',
                'home_form_ppg_5', 'away_form_ppg_5', 'home_days_rest', 'away_days_rest']

if USE_CLUB_POSITION:
    flag_sources += ['home_club_position', 'away_club_position']

for col in flag_sources:
    games[f'{col}_missing'] = games[col].isna().astype(int)

missing_cols = [f'{c}_missing' for c in flag_sources]

games['home_club_formation'] = games['home_club_formation'].fillna('unknown')
games['away_club_formation'] = games['away_club_formation'].fillna('unknown')

## Encode + scale

In [35]:
games['elo_diff'] = games['home_elo'] - games['away_elo']
games['manager_elo_diff'] = games['home_manager_elo'] - games['away_manager_elo']
games['value_ratio'] = np.log1p(games['home_squad_value']) - np.log1p(games['away_squad_value'])
games['form_diff_5'] = games['home_form_ppg_5'] - games['away_form_ppg_5']
games['form_diff_10'] = games['home_form_ppg_10'] - games['away_form_ppg_10']
games['season_ppg_diff'] = games['home_season_ppg'] - games['away_season_ppg']
games['rest_diff'] = games['home_days_rest'] - games['away_days_rest']

diff_cols = ['elo_diff', 'manager_elo_diff', 'value_ratio',
             'form_diff_5', 'form_diff_10', 'season_ppg_diff', 'rest_diff']

## Transform the UCL set here

Doing it here instead of later means it gets the exact same scaler as the training data.
I learned this the hard way.

In [36]:
games['result'] = np.select(
    [games['home_club_goals'] > games['away_club_goals'],
     games['home_club_goals'] < games['away_club_goals']],
    ['home_win', 'away_win'], default='draw'
)
print(games['result'].value_counts(normalize=True))

result
home_win    0.450661
away_win    0.331890
draw        0.217450
Name: proportion, dtype: float64


## Club names

So results say "Barcelona" not "131".

In [37]:
categorical_cols = ['competition_id', 'round', 'home_club_formation',
                    'away_club_formation', 'competition_type']

numeric_cols = [
    'home_elo', 'away_elo', 'home_manager_elo', 'away_manager_elo',
    'home_squad_value', 'away_squad_value', 'home_avg_value', 'away_avg_value',
    'home_lineup_stability', 'away_lineup_stability',
    'home_form_ppg_5', 'away_form_ppg_5', 'home_form_ppg_10', 'away_form_ppg_10',
    'home_form_gf_5', 'away_form_gf_5', 'home_form_ga_5', 'away_form_ga_5',
    'home_form_gf_10', 'away_form_gf_10', 'home_form_ga_10', 'away_form_ga_10',
    'home_season_ppg', 'away_season_ppg', 'home_season_played', 'away_season_played',
    'home_days_rest', 'away_days_rest',
] + diff_cols + tie_cols

if USE_CLUB_POSITION:
    numeric_cols += ['home_club_position', 'away_club_position']

keep = categorical_cols + numeric_cols + missing_cols + tie_flags
X = games[keep].copy()
y = games['result'].copy()

# metadata kept aligned with X so predictions can be traced back to real fixtures
meta = games[['game_id', 'date', 'season', 'competition_id', 'round',
              'home_club_id', 'away_club_id',
              'leg', 'tie_key', 'agg_lead', 'is_knockout']].copy()

print(X.shape)

(77342, 58)


## Save everything

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
meta_train, meta_test = train_test_split(meta, test_size=0.2, shuffle=False)

print(f"train: {meta_train['date'].min().date()} -> {meta_train['date'].max().date()}  ({len(X_train)})")
print(f"test : {meta_test['date'].min().date()} -> {meta_test['date'].max().date()}  ({len(X_test)})")

train: 2013-07-02 -> 2024-07-18  (61873)
test : 2024-07-18 -> 2026-07-06  (15469)


## Done

Now run Part 2.

In [39]:
train_medians = X_train[numeric_cols].median()

X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
X_test[numeric_cols] = X_test[numeric_cols].fillna(train_medians)

X_train['home_lineup_stability'] = X_train['home_lineup_stability'].fillna(0.5)
X_train['away_lineup_stability'] = X_train['away_lineup_stability'].fillna(0.5)
X_test['home_lineup_stability'] = X_test['home_lineup_stability'].fillna(0.5)
X_test['away_lineup_stability'] = X_test['away_lineup_stability'].fillna(0.5)

assert X_train[numeric_cols].isna().sum().sum() == 0
assert X_test[numeric_cols].isna().sum().sum() == 0

In [40]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

scaler = MinMaxScaler()
X_train_num = scaler.fit_transform(X_train[numeric_cols])
X_test_num = scaler.transform(X_test[numeric_cols])

X_train_final = np.hstack([X_train_cat, X_train_num,
                           X_train[missing_cols].values, X_train[tie_flags].values]).astype(np.float32)
X_test_final = np.hstack([X_test_cat,  X_test_num,
                           X_test[missing_cols].values,  X_test[tie_flags].values]).astype(np.float32)

feature_names = (list(encoder.get_feature_names_out(categorical_cols))
                 + numeric_cols + missing_cols + tie_flags)
print(X_train_final.shape, X_test_final.shape, len(feature_names))

(61873, 319) (15469, 319) 319


In [41]:
def transform(df):
    df = df.copy()
    df[numeric_cols] = df[numeric_cols].fillna(train_medians)
    df['home_lineup_stability'] = df['home_lineup_stability'].fillna(0.5)
    df['away_lineup_stability'] = df['away_lineup_stability'].fillna(0.5)
    return np.hstack([
        encoder.transform(df[categorical_cols]),
        scaler.transform(df[numeric_cols]),
        df[missing_cols].values,
        df[tie_flags].values
    ]).astype(np.float32)


ucl_mask  = (games['competition_id'] == 'CL') & (games['season'] == 2025)
ucl_games = games[ucl_mask]

X_ucl = transform(ucl_games[keep])
y_ucl = ucl_games['result'].values
meta_ucl = ucl_games[['game_id', 'date', 'round', 'home_club_id', 'away_club_id',
                      'leg', 'tie_key', 'agg_lead', 'is_knockout']]

# is any of it inside the training window? if so the evaluation is not held out
overlap = set(meta_ucl['game_id']) & set(meta_train['game_id'])
print(f"UCL matches: {len(X_ucl)}   leaked into train: {len(overlap)}")

UCL matches: 189   leaked into train: 0


In [42]:
club_names = pd.concat([
    games[['home_club_id', 'home_club_name']].rename(
        columns={'home_club_id': 'club_id', 'home_club_name': 'name'}),
    games[['away_club_id', 'away_club_name']].rename(
        columns={'away_club_id': 'club_id', 'away_club_name': 'name'}),
]).dropna().drop_duplicates('club_id')

club_names.to_csv('club_names.csv', index=False)
club_names.head()

,club_id,name
0,27843,Bala Town
1,28958,FC Lusitanos
2,4807,FC Shirak Gyumri
3,329,Sliema Wanderers
4,7130,JK Trans Narva


In [43]:
np.savez(
    'ucl_predictor_data.npz',
    X_train=X_train_final, X_test=X_test_final,
    y_train=y_train.values, y_test=y_test.values,
    X_ucl=X_ucl, y_ucl=y_ucl,
)

np.save('feature_names.npy', np.array(feature_names))

meta_train.to_csv('meta_train.csv', index=False)
meta_test.to_csv('meta_test.csv', index=False)
meta_ucl.to_csv('meta_ucl.csv', index=False)

joblib.dump({'encoder': encoder, 'scaler': scaler, 'train_medians': train_medians,
             'categorical_cols': categorical_cols, 'numeric_cols': numeric_cols,
             'missing_cols': missing_cols, 'tie_flags': tie_flags,
             'feature_names': feature_names},
            'preprocessing.joblib')

print('saved.')

saved.
